# 03a - Fair Propagator Comparison

**Pipeline stage 3a.** This notebook tests whether EnKF driver propagators other than random walk improve the downstream LSTM-EnKF model under a fair pipeline contract.

The legacy propagator sensitivity cell in notebook 05 swaps the EnKF forecast step at test time after the LSTM has already been trained. That is diagnostic only. This notebook rebuilds the full EnKF-preprocessed branch for each propagation method:

1. Fit the propagator on the training slice only.
2. Run causal EnKF preprocessing over the chronological driver series with that propagator.
3. Recompute onset-derived features and lag/rolling features from the preprocessed drivers.
4. Split, impute, scale, and build sequences with the tuned EnKF sequence length from notebook 02.
5. Train one LSTM per propagator with the same tuned hyperparameters and training budget.
6. Tune the classification threshold on validation predictions and report held-out test metrics.

**Outputs**
- Branch artifacts under `output_refactored/propagator_comparison/`.
- `PROPAGATOR_RESULTS_PATH` for notebooks 05/07 compatibility.
- `output/TableS4_propagator_sensitivity.csv` and `PROPAGATOR_COMPARISON_TABLE_PATH`.

Run after notebook 02 has produced `best_hps_lstm_enkf.json`. It can run before or after notebook 03 because it trains its own branch-specific models.

In [1]:
from shared_config import *
from propagator_comparison import run_fair_propagator_comparison

print_config_summary()
print(f"Propagator comparison output: {PROPAGATOR_COMPARISON_DIR}")

--- Workflow Configuration Summary ---
Enhanced Features Enabled:     True
Scaler Type Selected:          Robust
Class Weighting Enabled:       True
Hyperparameter Tuning Enabled: True
EnKF Enabled:                  True
EnKF Propagator:               SeasonalAR1
FAST_TEST Mode:                False
Sequence Length (default):     26
Forecast Horizon:              1 week(s)
Output directory:              output_refactored/
--------------------------------------
Propagator comparison output: output_refactored/propagator_comparison


## Run Comparison

This is the expensive cell. In production mode it trains four LSTM-EnKF branches: RandomWalk, AR1, VAR1, and SeasonalAR1. When `FAST_TEST=True`, the runner reduces EnKF members and training epochs for a smoke test.

In [2]:
results, table_s4, ranked = run_fair_propagator_comparison()

display(table_s4)
display(ranked[[
    "propagator", "acc", "prec", "rec", "f1", "brier",
    "auc", "auprc", "ohr", "threshold", "n_test"
]])


FAIR PROPAGATOR BRANCH: RandomWalk
EnKF Initialized. Members: 50, State Dim: 6
Sequences: X_train=(1097, 26, 56), X_val=(214, 26, 56), X_test=(216, 26, 56)

Epoch 1/50

Epoch 1: val_loss improved from None to 0.69230, saving model to output_refactored\propagator_comparison\randomwalk\lstm_enkf_propagator_randomwalk.keras

Epoch 1: finished saving model to output_refactored\propagator_comparison\randomwalk\lstm_enkf_propagator_randomwalk.keras
35/35 - 1s - 43ms/step - accuracy: 0.6408 - auc: 0.8001 - auprc: 0.5123 - loss: 1.1718 - precision: 0.3876 - recall: 0.8885 - val_accuracy: 0.7150 - val_auc: 0.8796 - val_auprc: 0.8398 - val_loss: 0.6923 - val_precision: 0.6039 - val_recall: 1.0000
Epoch 2/50

Epoch 2: val_loss did not improve from 0.69230
35/35 - 0s - 6ms/step - accuracy: 0.6299 - auc: 0.8599 - auprc: 0.6608 - loss: 1.0422 - precision: 0.3841 - recall: 0.9308 - val_accuracy: 0.7430 - val_auc: 0.8642 - val_auprc: 0.8253 - val_loss: 0.7002 - val_precision: 0.6284 - val_recall: 1.0

,Propagator,Accuracy (%),Precision (%),Recall (%),F1 (%),Brier Score,Onset Hit Rate,Threshold,Q trace,VAR(1) rho(A)
0,RandomWalk *,74.07,55.56,100.00,71.43,0.1490,9/9 (100%),0.399,9.774e+13,NaN
1,AR1,84.26,68.00,97.14,80.00,0.1482,8/9 (89%),0.604,6.003e+13,NaN
2,VAR1,76.85,58.77,95.71,72.83,0.1847,7/9 (78%),0.606,5.961e+13,0.808
3,SeasonalAR1,68.98,51.11,98.57,67.32,0.1679,8/9 (89%),0.445,5.978e+13,NaN


,propagator,acc,prec,rec,f1,brier,auc,auprc,ohr,threshold,n_test
1,AR1,0.842593,0.680000,0.971429,0.800000,0.148203,0.936399,0.873888,0.888889,0.604446,216
0,RandomWalk,0.740741,0.555556,1.000000,0.714286,0.148985,0.930431,0.875745,1.000000,0.398916,216
3,SeasonalAR1,0.689815,0.511111,0.985714,0.673171,0.167892,0.936791,0.882791,0.888889,0.444589,216
2,VAR1,0.768519,0.587719,0.957143,0.728261,0.184678,0.932877,0.887855,0.777778,0.605774,216
